# LTH-SOPR Exit Signal Analysis

**Hypothesis:** Long-Term Holder SOPR > 1 signals distribution/tops better than STH-SOPR.

**Why LTH-SOPR might work better for exits:**
1. LTHs rarely sell - when they do, it's significant
2. LTHs are "smart money" - they know cycles
3. Less noise than STH-SOPR
4. LTH selling at profit = real distribution, not just trading

**The Framework:**
- Entry: STH-SOPR < 1 (short-term holders capitulating)
- Exit: LTH-SOPR > 1.x (long-term holders distributing)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from numba import njit
import warnings
warnings.filterwarnings('ignore')

print("LTH-SOPR Exit Signal Analysis 📊")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
sopr_lth = pd.read_parquet(DATA_DIR / "sopr_lth.parquet").rename(columns={"value": "sopr_lth"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(sopr_lth, how='inner').join(mvrv, how='inner').join(realized_loss, how='inner')
df = df.sort_index()
df = df[df.index >= '2019-01-01'].dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")
print(f"\nLTH-SOPR stats:")
print(f"  Min: {df['sopr_lth'].min():.3f}")
print(f"  Max: {df['sopr_lth'].max():.3f}")
print(f"  Mean: {df['sopr_lth'].mean():.3f}")
print(f"  % > 1: {(df['sopr_lth'] > 1).mean()*100:.1f}%")

In [ ]:
# Calculate z-scores and derived metrics
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
df['lth_sopr_z14'] = (df['sopr_lth'] - df['sopr_lth'].rolling(14).mean()) / df['sopr_lth'].rolling(14).std()
df['lth_sopr_z30'] = (df['sopr_lth'] - df['sopr_lth'].rolling(30).mean()) / df['sopr_lth'].rolling(30).std()

# Forward returns
df['fwd_30d'] = df['price'].shift(-30) / df['price'] - 1
df['fwd_90d'] = df['price'].shift(-90) / df['price'] - 1

df = df.dropna()
print(f"Data after calculations: {len(df)} rows")

---
## 1. LTH-SOPR vs STH-SOPR Comparison

In [ ]:
# Compare signal frequency and behavior
print("SIGNAL FREQUENCY COMPARISON")
print("="*60)
print(f"\n{'Condition':<25} {'STH-SOPR':>15} {'LTH-SOPR':>15}")
print("-"*60)

conditions = [
    ('> 1.00', 1.00),
    ('> 1.02', 1.02),
    ('> 1.05', 1.05),
    ('> 1.10', 1.10),
    ('> 1.20', 1.20),
    ('> 1.50', 1.50),
    ('> 2.00', 2.00),
]

for label, thresh in conditions:
    sth_pct = (df['sopr_sth'] > thresh).mean() * 100
    lth_pct = (df['sopr_lth'] > thresh).mean() * 100
    print(f"{label:<25} {sth_pct:>14.1f}% {lth_pct:>14.1f}%")

In [ ]:
# Visualize both
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.4, 0.2, 0.2, 0.2],
    subplot_titles=('BTC Price (Log)', 'STH-SOPR', 'LTH-SOPR', 'MVRV')
)

fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price', line=dict(color='orange')), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['sopr_sth'], name='STH-SOPR', line=dict(color='blue')), row=2, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='gray', row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['sopr_lth'], name='LTH-SOPR', line=dict(color='purple')), row=3, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='gray', row=3, col=1)
fig.add_hline(y=1.5, line_dash='dash', line_color='red', row=3, col=1)
fig.add_hline(y=2.0, line_dash='dash', line_color='darkred', row=3, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['mvrv'], name='MVRV', line=dict(color='green')), row=4, col=1)
fig.add_hline(y=2, line_dash='dash', line_color='orange', row=4, col=1)
fig.add_hline(y=2.5, line_dash='dash', line_color='red', row=4, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=1000, title='STH-SOPR vs LTH-SOPR Comparison', showlegend=False)
fig.show()

---
## 2. Forward Returns Analysis

In [ ]:
print("FORWARD RETURNS BY EXIT SIGNAL")
print("="*110)
print(f"{'Signal':<40} {'Avg 30d':>12} {'Avg 90d':>12} {'Days':>10} {'% Time':>10}")
print("-"*110)

exit_signals = [
    ('All days', df['sopr'] > 0),
    
    # STH-SOPR (baseline)
    ('STH-SOPR > 1.00', df['sopr_sth'] > 1.00),
    ('STH-SOPR > 1.02', df['sopr_sth'] > 1.02),
    ('STH-SOPR > 1.05', df['sopr_sth'] > 1.05),
    
    # LTH-SOPR
    ('LTH-SOPR > 1.00', df['sopr_lth'] > 1.00),
    ('LTH-SOPR > 1.02', df['sopr_lth'] > 1.02),
    ('LTH-SOPR > 1.05', df['sopr_lth'] > 1.05),
    ('LTH-SOPR > 1.10', df['sopr_lth'] > 1.10),
    ('LTH-SOPR > 1.20', df['sopr_lth'] > 1.20),
    ('LTH-SOPR > 1.50', df['sopr_lth'] > 1.50),
    ('LTH-SOPR > 2.00', df['sopr_lth'] > 2.00),
    
    # LTH-SOPR Z-score
    ('LTH-SOPR Z14 > 1.0', df['lth_sopr_z14'] > 1.0),
    ('LTH-SOPR Z14 > 1.5', df['lth_sopr_z14'] > 1.5),
    ('LTH-SOPR Z14 > 2.0', df['lth_sopr_z14'] > 2.0),
    
    # MVRV (comparison)
    ('MVRV > 2.0', df['mvrv'] > 2.0),
    ('MVRV > 2.5', df['mvrv'] > 2.5),
    
    # Combined
    ('MVRV>2 + LTH>1.05', (df['mvrv'] > 2) & (df['sopr_lth'] > 1.05)),
    ('MVRV>2 + LTH>1.10', (df['mvrv'] > 2) & (df['sopr_lth'] > 1.10)),
    ('MVRV>2 + LTH>1.20', (df['mvrv'] > 2) & (df['sopr_lth'] > 1.20)),
    ('MVRV>2.5 + LTH>1.10', (df['mvrv'] > 2.5) & (df['sopr_lth'] > 1.10)),
    ('MVRV>2.5 + LTH>1.50', (df['mvrv'] > 2.5) & (df['sopr_lth'] > 1.50)),
]

results = []
for name, cond in exit_signals:
    subset = df[cond]
    if len(subset) > 10:
        avg_30 = subset['fwd_30d'].mean() * 100
        avg_90 = subset['fwd_90d'].mean() * 100
        pct = len(subset) / len(df) * 100
        results.append({'name': name, 'avg_30': avg_30, 'avg_90': avg_90, 'days': len(subset), 'pct': pct})
        marker = '⭐' if avg_90 < 0 else ''
        print(f"{name:<40} {avg_30:>+11.1f}% {avg_90:>+11.1f}% {len(subset):>10} {pct:>9.1f}% {marker}")

In [ ]:
# Rank by most negative forward returns
print("\n" + "="*70)
print("BEST EXIT SIGNALS (Most Negative 90d Forward Returns)")
print("="*70)

sorted_results = sorted(results, key=lambda x: x['avg_90'])
for i, r in enumerate(sorted_results[:15]):
    quality = '⭐' if r['avg_90'] < 0 else ''
    print(f"{i+1:>2}. {r['name']:<40} {r['avg_90']:>+8.1f}% ({r['pct']:.1f}% of time) {quality}")

---
## 3. Known Tops Analysis

In [ ]:
# What was LTH-SOPR at known tops?
major_tops = [
    ('2021-04-14', 'April 2021 ATH'),
    ('2021-11-10', 'Nov 2021 ATH'),
    ('2024-03-14', 'March 2024 ATH'),
    ('2024-12-17', 'Dec 2024 ATH'),
    ('2021-05-10', 'Pre-China Crash'),
    ('2022-03-28', 'Bear Rally 1'),
]

print("LTH-SOPR AT MAJOR TOPS")
print("="*120)
print(f"{'Date':<12} {'Event':<20} {'Price':>10} {'STH-SOPR':>10} {'LTH-SOPR':>10} {'MVRV':>8} {'30d Fwd':>10}")
print("-"*100)

top_data = []
for date_str, name in major_tops:
    try:
        target = pd.Timestamp(date_str, tz='UTC')
        idx = df.index.get_indexer([target], method='nearest')[0]
        row = df.iloc[idx]
        actual_date = df.index[idx]
        
        top_data.append({
            'date': actual_date,
            'name': name,
            'sth_sopr': row['sopr_sth'],
            'lth_sopr': row['sopr_lth'],
            'mvrv': row['mvrv'],
            'fwd_30d': row['fwd_30d'],
        })
        
        fwd = f"{row['fwd_30d']*100:+.0f}%" if pd.notna(row['fwd_30d']) else 'N/A'
        print(f"{actual_date.strftime('%Y-%m-%d'):<12} {name:<20} ${row['price']:>9,.0f} {row['sopr_sth']:>10.3f} {row['sopr_lth']:>10.3f} {row['mvrv']:>8.2f} {fwd:>10}")
    except Exception as e:
        print(f"Error: {e}")

tops_df = pd.DataFrame(top_data)

print(f"\nSummary:")
print(f"  Avg STH-SOPR at tops: {tops_df['sth_sopr'].mean():.3f}")
print(f"  Avg LTH-SOPR at tops: {tops_df['lth_sopr'].mean():.3f}")
print(f"  % with LTH-SOPR > 1.0: {(tops_df['lth_sopr'] > 1.0).mean()*100:.0f}%")
print(f"  % with LTH-SOPR > 1.5: {(tops_df['lth_sopr'] > 1.5).mean()*100:.0f}%")
print(f"  % with LTH-SOPR > 2.0: {(tops_df['lth_sopr'] > 2.0).mean()*100:.0f}%")

---
## 4. Backtest LTH-SOPR Exit Strategy

In [ ]:
@njit
def exit_simple_trail(price_arr, sth_arr, lth_arr, mvrv_arr, entry_idx, trail_pct=0.30):
    """Baseline: Simple trailing stop"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price > peak:
            peak = price
        if price <= peak * (1 - trail_pct):
            return j, price, 'trail'
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_lth_sopr(price_arr, sth_arr, lth_arr, mvrv_arr, entry_idx,
                  mvrv_context=2.0, lth_exit=1.10,
                  trail_before=0.30, trail_after=0.20):
    """
    LTH-SOPR exit strategy:
    - Before trigger: wide trail
    - After MVRV + LTH-SOPR trigger: tighter trail
    """
    entry_price = price_arr[entry_idx]
    peak = entry_price
    triggered = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        lth = lth_arr[j]
        mvrv = mvrv_arr[j]
        
        if price > peak:
            peak = price
        
        # Check for LTH distribution trigger
        if not triggered:
            if mvrv > mvrv_context and lth > lth_exit:
                triggered = True
        
        trail = trail_after if triggered else trail_before
        
        if price <= peak * (1 - trail):
            reason = 'lth_trail' if triggered else 'trail'
            return j, price, reason
    
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_lth_hard(price_arr, sth_arr, lth_arr, mvrv_arr, entry_idx,
                  mvrv_context=2.0, lth_exit=1.50, trail_pct=0.30):
    """
    Hard exit when LTH-SOPR spikes (smart money selling)
    """
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        lth = lth_arr[j]
        mvrv = mvrv_arr[j]
        
        if price > peak:
            peak = price
        
        # Hard exit on LTH distribution
        if mvrv > mvrv_context and lth > lth_exit:
            return j, price, 'lth_hard'
        
        # Fallback trail
        if price <= peak * (1 - trail_pct):
            return j, price, 'trail'
    
    return len(price_arr) - 1, price_arr[-1], 'hold'

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, **kwargs):
    price_arr = df['price'].values
    sth_arr = df['sopr_sth'].values
    lth_arr = df['sopr_lth'].values
    mvrv_arr = df['mvrv'].values
    dates = df.index
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        exit_idx, exit_price, exit_reason = exit_func(price_arr, sth_arr, lth_arr, mvrv_arr, entry_idx, **kwargs)
        
        entry_price = price_arr[entry_idx]
        net_return = (exit_price / entry_price) - 1 - 0.002
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'exit_reason': exit_reason,
            'exit_lth': lth_arr[exit_idx],
            'exit_mvrv': mvrv_arr[exit_idx],
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    return trades_df


def calc_metrics(trades, initial_capital=100000):
    if len(trades) == 0:
        return None
    final = trades['equity'].iloc[-1]
    total_ret = (final / initial_capital) - 1
    years = (trades['exit_date'].iloc[-1] - trades['entry_date'].iloc[0]).days / 365.25
    win_rate = (trades['net_return'] > 0).mean()
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(trades)/years) if returns.std() > 0 and years > 0 else 0
    
    equity = [initial_capital] + list(trades['equity'])
    peak, max_dd = equity[0], 0
    for eq in equity:
        if eq > peak: peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd: max_dd = dd
    
    return {
        'total_return': total_ret,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'win_rate': win_rate,
        'n_trades': len(trades),
        'avg_hold': trades['days_held'].mean()
    }

In [ ]:
# Entry: STH-SOPR < 1 (for STRAT-003 comparison)
entry_cond = df['sopr_sth'] < 1
entries = entry_cond & ~entry_cond.shift(1).fillna(False)

print("LTH-SOPR EXIT STRATEGY BACKTEST")
print("Entry: STH-SOPR < 1")
print("="*120)

strategies = [
    # Baselines
    ('Simple 30% Trail', exit_simple_trail, {'trail_pct': 0.30}),
    ('Simple 20% Trail', exit_simple_trail, {'trail_pct': 0.20}),
    ('Simple 15% Trail', exit_simple_trail, {'trail_pct': 0.15}),
    ('Simple 8% Trail', exit_simple_trail, {'trail_pct': 0.08}),
    
    # LTH-SOPR triggered trail
    ('LTH>1.05 + MVRV>2 → 30/20', exit_lth_sopr,
     {'mvrv_context': 2.0, 'lth_exit': 1.05, 'trail_before': 0.30, 'trail_after': 0.20}),
    ('LTH>1.10 + MVRV>2 → 30/20', exit_lth_sopr,
     {'mvrv_context': 2.0, 'lth_exit': 1.10, 'trail_before': 0.30, 'trail_after': 0.20}),
    ('LTH>1.20 + MVRV>2 → 30/15', exit_lth_sopr,
     {'mvrv_context': 2.0, 'lth_exit': 1.20, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('LTH>1.50 + MVRV>2 → 30/15', exit_lth_sopr,
     {'mvrv_context': 2.0, 'lth_exit': 1.50, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('LTH>1.10 + MVRV>2.5 → 30/15', exit_lth_sopr,
     {'mvrv_context': 2.5, 'lth_exit': 1.10, 'trail_before': 0.30, 'trail_after': 0.15}),
    
    # LTH-SOPR hard exit
    ('LTH>1.50 hard exit', exit_lth_hard,
     {'mvrv_context': 2.0, 'lth_exit': 1.50, 'trail_pct': 0.30}),
    ('LTH>2.00 hard exit', exit_lth_hard,
     {'mvrv_context': 2.0, 'lth_exit': 2.00, 'trail_pct': 0.30}),
]

print(f"{'Strategy':<35} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'Trades':>7} {'AvgHold':>8}")
print("-"*100)

all_results = []
for name, func, kwargs in strategies:
    trades = run_backtest(df, entries, func, **kwargs)
    m = calc_metrics(trades)
    if m:
        print(f"{name:<35} {m['total_return']*100:>+9.0f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {m['n_trades']:>7} {m['avg_hold']:>7.0f}d")
        all_results.append({'name': name, 'metrics': m, 'trades': trades})

In [ ]:
# Compare best LTH vs best simple
best_simple = max([r for r in all_results if 'Simple' in r['name']], key=lambda x: x['metrics']['total_return'])
best_lth = max([r for r in all_results if 'LTH' in r['name']], key=lambda x: x['metrics']['total_return'])

print("\n" + "="*70)
print("BEST SIMPLE vs BEST LTH-SOPR")
print("="*70)

print(f"\n{'Metric':<20} {'Simple':>20} {'LTH-SOPR':>20} {'Diff':>15}")
print("-"*75)

ms, ml = best_simple['metrics'], best_lth['metrics']
for label, key, mult in [('Return', 'total_return', 100), ('Sharpe', 'sharpe', 1), ('Max DD', 'max_dd', 100), ('Win Rate', 'win_rate', 100)]:
    s, l = ms[key] * mult, ml[key] * mult
    diff = l - s
    suffix = '%' if mult == 100 else ''
    print(f"{label:<20} {s:>19.1f}{suffix} {l:>19.1f}{suffix} {diff:>+14.1f}{suffix}")

print(f"\nSimple: {best_simple['name']}")
print(f"LTH-SOPR: {best_lth['name']}")
print(f"\n🏆 WINNER: {'LTH-SOPR' if ml['total_return'] > ms['total_return'] else 'Simple Trail'}")

In [ ]:
# Trade log for best LTH strategy
print("\n" + "="*100)
print(f"TRADE LOG: {best_lth['name']}")
print("="*100)

t = best_lth['trades']
print(f"\n{'Entry':<12} {'Exit':<12} {'Days':>6} {'Return':>10} {'Reason':<12} {'LTH':>8} {'MVRV':>8}")
print("-"*80)

for _, row in t.iterrows():
    print(f"{row['entry_date'].strftime('%Y-%m-%d'):<12} {row['exit_date'].strftime('%Y-%m-%d'):<12} {row['days_held']:>6} {row['net_return']*100:>+9.0f}% {row['exit_reason']:<12} {row['exit_lth']:>8.2f} {row['exit_mvrv']:>8.2f}")

---
## 5. Test with STRAT-002 Entry (Long-term)

In [ ]:
# Full STRAT-002 entry
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
entry_cond_full = (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['rl_zscore'] > 0.5)
entries_full = entry_cond_full & ~entry_cond_full.shift(1).fillna(False)

print("\nSTRAT-002 ENTRY + LTH-SOPR EXIT")
print("Entry: SOPR < 1 AND STH-SOPR < 1 AND RL Z > 0.5")
print("="*100)

strat2_tests = [
    ('Simple 30% Trail (v5)', exit_simple_trail, {'trail_pct': 0.30}),
    ('LTH>1.10 + MVRV>2 → 30/20', exit_lth_sopr,
     {'mvrv_context': 2.0, 'lth_exit': 1.10, 'trail_before': 0.30, 'trail_after': 0.20}),
    ('LTH>1.20 + MVRV>2 → 30/15', exit_lth_sopr,
     {'mvrv_context': 2.0, 'lth_exit': 1.20, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('LTH>1.50 + MVRV>2.5 → 30/15', exit_lth_sopr,
     {'mvrv_context': 2.5, 'lth_exit': 1.50, 'trail_before': 0.30, 'trail_after': 0.15}),
]

print(f"{'Strategy':<35} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'Trades':>7}")
print("-"*85)

for name, func, kwargs in strat2_tests:
    trades = run_backtest(df, entries_full, func, **kwargs)
    m = calc_metrics(trades)
    if m:
        print(f"{name:<35} {m['total_return']*100:>+9.0f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {m['n_trades']:>7}")

---
## 6. Summary

In [ ]:
print("\n" + "="*70)
print("LTH-SOPR EXIT SIGNAL SUMMARY")
print("="*70)

print(f"""
📊 LTH-SOPR CHARACTERISTICS:
   • LTH-SOPR > 1.0 occurs {(df['sopr_lth'] > 1).mean()*100:.0f}% of time (vs STH: {(df['sopr_sth'] > 1).mean()*100:.0f}%)
   • LTH-SOPR > 1.5 occurs {(df['sopr_lth'] > 1.5).mean()*100:.0f}% of time
   • LTH-SOPR > 2.0 occurs {(df['sopr_lth'] > 2.0).mean()*100:.0f}% of time
   • Much higher values than STH-SOPR (LTHs have bigger cost basis gap)

📈 AT MAJOR TOPS:
   • Avg STH-SOPR: {tops_df['sth_sopr'].mean():.2f}
   • Avg LTH-SOPR: {tops_df['lth_sopr'].mean():.2f}

💡 KEY INSIGHT:
   • LTH-SOPR has wider range (can go > 2.0) vs STH-SOPR (~1.0-1.1)
   • Higher thresholds are more meaningful (true "smart money" distribution)
   • But also rarer - may not trigger in shorter cycles

🏆 RESULTS:
   Simple Trail: {best_simple['metrics']['total_return']*100:+,.0f}%
   LTH-SOPR Exit: {best_lth['metrics']['total_return']*100:+,.0f}%
   Winner: {'LTH-SOPR ✅' if best_lth['metrics']['total_return'] > best_simple['metrics']['total_return'] else 'Simple Trail ✅'}
""")